# Ariadne — FitVision AI | LoRA One-Click Training (Flux/SDXL)
**Dataset:** 20–50 multi-angle images (lovechara.work / tanuki-kimono / @LolitaWardrobe) + .txt captions sidecar  
**Output:** `pytorch_lora_weights.safetensors` + TensorBoard loss curve for COMPFEST Before/After evidence  
**Runtime:** T4 / A100 GPU • No local GPU needed

> After training, upload `safetensors` to HuggingFace and point `lib/replicate.ts` to your custom Replicate deployment.


## 0) Check GPU


In [ ]:
!nvidia-smi
import torch; print("cuda available:", torch.cuda.is_available(), "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

## 1) Install — diffusers + peft + accelerate


In [ ]:
!pip install -q diffusers[torch] peft accelerate transformers wandb tensorboard safetensors pillow
!pip install -q git+https://github.com/huggingface/diffusers.git


## 2) Clone Ariadne + prepare dataset
Upload your 20–50 images as a ZIP (images + .txt captions) **or** mount Drive.


In [ ]:
!git clone https://github.com/OselC/Fitvision-AI.git /content/Ariadne 2>/dev/null || echo "Ariadne exists"
%cd /content/Ariadne
!ls -R ai-pipeline/lora

# Option A: Mount Drive and copy datasets/lolita_01
# from google.colab import drive; drive.mount('/content/drive')
# !cp -r "/content/drive/MyDrive/ariadne/datasets/lolita_01" ./datasets/

# Option B: Upload ZIP manually then:
# !unzip -q /content/lolita_01.zip -d ./datasets/
!mkdir -p ./datasets/lolita_01 ./weights/lolita-lora
!echo "Datasets ready — put 20-50 jpg + txt in ./datasets/lolita_01" && ls -lh ./datasets/lolita_01 | head -n 20

### Quick check: dataset must have .txt captions (otherwise auto caption used)


In [ ]:
!ls -1 ./datasets/lolita_01 | head -n 20
!echo "--- captions sample ---" && cat ./datasets/lolita_01/*.txt 2>/dev/null | head -n 6 || echo "(no .txt — fallback prompt will be used)"

## 3) One-click LoRA training (Flux.1-dev, rank 16, 1500 steps)
Edit args below if you use SDXL instead: change `--pretrained_model_name_or_path` to `stabilityai/stable-diffusion-xl-base-1.0`


In [ ]:
!accelerate launch --mixed_precision=fp16 ai-pipeline/lora/train_lora.py \
  --pretrained_model_name_or_path="black-forest-labs/FLUX.1-dev" \
  --dataset_dir="./datasets/lolita_01" \
  --output_dir="./weights/lolita-lora" \
  --rank=16 --learning_rate=1e-4 --max_train_steps=1500 \
  --train_batch_size=1 --gradient_accumulation_steps=4 --resolution=1024 \
  --report_to=tensorboard

## 4) Check outputs + loss curve


In [ ]:
!ls -lh ./weights/lolita-lora/
!echo "--- TensorBoard logs ---" && ls -lh ./weights/lolita-lora/logs 2>/dev/null || ls -lh runs 2>/dev/null | head

# Launch TensorBoard inline
%load_ext tensorboard
%tensorboard --logdir ./weights/lolita-lora --host 0.0.0.0 --port 6006

# To download weights:
# from google.colab import files; !zip -r lora.zip ./weights/lolita-lora/*.safetensors && files.download('lora.zip')

## 5) Before vs After collage for pitch deck


In [ ]:
# Generate two renders: before (base Flux) vs after (with LoRA) — replace with your real test images
!python ai-pipeline/eval/compare.py --before ai-pipeline/eval/before.jpg --after ai-pipeline/eval/after.jpg --out ./weights/lolita-lora/before_after.png
!ls -lh ./weights/lolita-lora/before_after.png
from IPython.display import Image; Image("./weights/lolita-lora/before_after.png")

### Next: Upload to HuggingFace and deploy on Replicate
```bash
# huggingface-cli login
# huggingface-cli upload your-username/ariadne-lolita-lora ./weights/lolita-lora/*.safetensors
```
Then create Replicate deployment that loads `black-forest-labs/FLUX.1-dev` + your LoRA and update `lib/replicate.ts:19` to `your-username/ariadne-lolita-lora`
